In [2]:
import pandas as pd


In [3]:

data_url = "https://raw.githubusercontent.com/fenago/tf/main/Chapter6-Regularization_and_Hyperparameter_Tuning/dataset/shuttle.trn"

data = pd.read_table(data_url, header=None, sep=' ')
data.head()

,0,1,2,3,4,5,6,7,8,9
0,50,21,77,0,28,0,27,48,22,2
1,55,0,92,0,0,26,36,92,56,4
2,53,0,82,0,52,-5,29,30,2,1
3,37,0,76,0,28,18,40,48,8,1
4,37,0,79,0,34,-26,43,46,2,1


In [4]:
y = data.pop(9)
X = data.copy()

In [5]:
from sklearn.model_selection import train_test_split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y)

In [8]:
!pip install keras-tuner
import keras_tuner as kt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 8.9 MB/s eta 0:00:00


In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Dense

In [10]:
tf.random.set_seed(8)


In [25]:
def model_builder(hp):
        model = tf.keras.Sequential()
        hp_units = hp.Int('units', min_value=128, max_value=512, \
                          step=64)
        reg_fc1 = Dense(hp_units, input_shape=(9,), \
                        activation='relu', \
                        kernel_regularizer=tf.keras.regularizers\
                                             .l2(l2=0.0001)) # Changed l to l2
        reg_fc2 = Dense(512, activation='relu', \
                        kernel_regularizer=tf.keras.regularizers\
                                             .l2(l2=0.0001)) # Changed l to l2
        reg_fc3 = Dense(128, activation='relu', \
                        kernel_regularizer=tf.keras.regularizers\
                                             .l2(l2=0.0001)) # Changed l to l2
        reg_fc4 = Dense(128, activation='relu', \
                        kernel_regularizer=tf.keras.regularizers\
                                             .l2(l2=0.0001)) # Changed l to l2
        reg_fc5 = Dense(8, activation='softmax')
        model.add(reg_fc1)
        model.add(reg_fc2)
        model.add(reg_fc3)
        model.add(reg_fc4)
        model.add(reg_fc5)
        loss = tf.keras.losses.SparseCategoricalCrossentropy()
        hp_learning_rate = hp.Choice('learning_rate', \
                                     values = [0.01, 0.001, 0.0001])
        optimizer = tf.keras.optimizers.Adam(hp_learning_rate)
        model.compile(optimizer = optimizer, loss = loss, \
                      metrics = ['accuracy'])
        return model

In [26]:
tuner = kt.Hyperband(model_builder, objective='val_accuracy', \
                            max_epochs=5, overwrite=True)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [27]:
tuner.search(X_train, y_train, validation_data=(X_test, y_test))

Trial 10 Complete [00h 00m 51s]
val_accuracy: 0.9921072721481323

Best val_accuracy So Far: 0.9973180294036865
Total elapsed time: 00h 05m 04s


In [28]:
best_hps = tuner.get_best_hyperparameters()[0]


In [29]:
best_units = best_hps.get('units')
best_units

128

In [30]:
best_lr = best_hps.get('learning_rate')
best_lr

0.001

In [31]:
model = tuner.hypermodel.build(best_hps)
model.fit(X_train, y_train, epochs=5, \
              validation_data=(X_test, y_test))

Epoch 1/5


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


952/952 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9479 - loss: 0.4327 - val_accuracy: 0.9828 - val_loss: 0.1783
Epoch 2/5
952/952 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9919 - loss: 0.1183 - val_accuracy: 0.9841 - val_loss: 0.1333
Epoch 3/5
952/952 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9943 - loss: 0.0707 - val_accuracy: 0.9974 - val_loss: 0.0985
Epoch 4/5
952/952 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9936 - loss: 0.0726 - val_accuracy: 0.9973 - val_loss: 0.0977
Epoch 5/5
952/952 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9946 - loss: 0.0617 - val_accuracy: 0.9975 - val_loss: 0.0654
